---
title: "Building an ML Platform"
description: "A small, low-ops ML platform built twice: first as a Docker Compose proof of concept, then ported to Azure Container Apps behind an explicit environment contract."
image: "./img/ml-platform-cover.png"
---

A step-by-step build of a **deliberately small, low-ops** MLOps platform for a
team of ML engineers who are *not* full-time platform/DevOps engineers. We use
the fewest moving parts that still deliver reproducible training, honest
evaluation, scheduled and on-demand batch/inference workflows, and observable
operations — and we add machinery only when a concrete need forces it.

The runnable source lives in **`projects/ml-platform/`**, independent of these
notebooks. Each chapter develops one slice of that project and then references
the modules and scripts it produced. The production contract the build follows
lives in **`projects/ml-platform/docs/`** (documents `00`–`08`).

## The learning path: one platform, two environments

The platform is built twice on purpose. Part I builds every feature on a
Docker Compose sandbox: Postgres and MinIO underneath, self-hosted MLflow, a
small runner service playing the execution plane, serving, dashboard, and the
local LLM commands. The feedback loop stays minutes long while the design
settles. Part II ports the settled result to Azure: a Terraform foundation,
Container Apps Jobs as the execution plane, managed Postgres and Blob storage,
then CI/CD and operations. Nothing conceptual changes in the move; only the
deployment adapters do.

| Part | Chapters | Environment | Outcome |
|---|---|---|---|
| I — Local platform | 02–08 | Docker Compose | Every feature working, contracts pinned |
| II — Production | 09–13 | Azure Container Apps + Terraform | Same workload images deployed and operated |
| Off the critical path | 14–15 | Azure ML / Redis broker | Exception tracks, adopted only if forced |
| Capstone | 16 | Compose + Azure adapters | Separate suites preserve one behavioral definition of done |

| Ch | Chapter | Builds in projects/ml-platform/ | Contract |
|----|---------|-----------------------------------|----------|
| 01 | Overview & the golden path | — (conceptual) | docs/00, docs/07 |
| 02 | Local platform foundation | demo/ | docs/01 |
| 03 | Reproducible training & registry | src/train_job/, src/ml_platform/common/ | docs/02 |
| 04 | Results DB & batch workflows | src/ml_platform/results/, src/batch_job/ | docs/04 |
| 05 | Online serving & promotion | src/serving_app/ | docs/05, docs/06 |
| 06 | Observability & dashboard | src/dashboard/ | docs/06 |
| 07 | LLM release artifacts | src/ml_platform/llm/ | docs/03 |
| 08 | The environment contract | tools/check_env_contract.py | all |
| 09 | Just enough Terraform | infra/ | docs/01 |
| 10 | Azure foundation | infra/, src/mlflow_app/ | docs/01 |
| 11 | Porting the workflows to ACA | deploy/, job definitions | docs/02–docs/05 |
| 12 | CI/CD | .github/workflows/ | docs/07 |
| 13 | Azure operations | alerts, dashboards, runbooks | docs/06 |
| 14 | Multi-GPU training | src/train_aml/ | docs/08 |
| 15 | Broker upgrade | conditional | docs/04 |
| 16 | End-to-end integration | demo/golden_path.py, deploy/smoke-tests.* | all |

Chapters 14 and 15 are off the critical path: an exception track and a
conditional upgrade, included so the boundary is explicit but never required to
ship Part I.


## Four planes, no control plane

Everything in the platform is a consequence of four planes plus a thin dashboard.

| Plane | Responsibility | Azure building block |
|---|---|---|
| Execution | Run every workflow as an ephemeral, image-pinned task | Azure Container Apps Jobs |
| Model lifecycle | Track experiments, register versions, store artifacts | Self-hosted MLflow (ACA App + Postgres + Blob) |
| Operational state | Record status/output/error for every run, with batch granularity | Generic results DB (Postgres) |
| Serving | Optional online HTTP inference at an exact model version | Azure Container Apps Apps |

There is no bespoke control plane: no Durable Functions, no orchestration engine,
and no application broker in the baseline. Linear multi-step workflows are an
ordinary Python script inside one Job; batch fan-out is expressed as parent/child
rows in the results DB plus a small stateless rule.

## One contract, two deployment adapters

What makes the Part II move a port and not a rewrite is a short list of seams,
fixed once in chapter 08:

- Environment variables: every service declares which variables it reads, and
  tools/check_env_contract.py fails when a variable has no provider or documented
  production injector.
- Workload entrypoints: Compose and ACA reference the same train, batch, serving,
  dashboard, and shared LLM image sources.
- Results DB: the same schema records every run in both worlds.
- Promotion semantics: flip the MLflow production alias, then repin only the
  long-running consumer to the exact version.
- Behavioral checks: the local golden path and cloud smoke adapters use separate
  trigger mechanisms but assert terminal success, results state, readiness, model
  identity, and prediction behavior.

Part I stands each plane up on Compose equivalents; Part II moves them onto Azure
blocks and managed identities without adding local-only feature implementations.


## How the course uses the project

The **source is not authored inside the notebooks**. Each chapter:

1. States the outcome and the slice of `projects/ml-platform/docs/` it implements.
2. Builds the relevant modules/scripts under `projects/ml-platform/src/` (and
   `demo/` in Part I, `infra/` / `deploy/` in Part II).
3. References that source when demonstrating a stage — e.g. triggering a
   training run through the runner API, or loading a model version in the
   serving app.
4. Ends with an **Extensions** section: what the production contract asks for
   that the chapter's MVP intentionally defers, and where it is specified.

This keeps the platform a real, reviewable project you could lift out of the
course, while the notebooks stay the narrated build log.

## Prerequisites & cost

- Part I: Docker and Python 3.11+, basic ML familiarity. No cloud account.
- Part II: Azure CLI with an active subscription; container builds via ACR
  Tasks (no local Docker required there).
- Budget is small with tear-down discipline; every Part II chapter that
  provisions resources ends by tearing them down.
